## Abstract
This quantitative question investigates whether the volume of preserved and shared digital cultural heritage on Europeana reflects the physical density of Galleries, Libraries, Archives, and Museums (GLAMs) or the scale of public cultural investment (% of GDP via Eurostat). We merge SPARQL queries from Wikidata and Europeana with offline administrative census data across six European nations. Our analysis uncovers a critical paradox: government funding does not linearly correlate with digital volume.

### Research questions


*  Is there a statistically significant correlation between government cultural expenditure (% of GDP) and the total volume of digitized cultural heritage shared on Europeana?
* To what extent does a nation's physical GLAM density predict the number of active, data-contributing providers on Europeana?

### Sources


1.   **Europeana SPARQL API / Portal Counts**: Total digital items shared per country (`europeana_total`).
2.   **Europeana Active Providers Database**: Number of active data providers contributing (`europeana_providers`).
3. **Eurostat (COFOG)**: Public expenditure on cultural services as a percentage of national GDP (`culture_expenditure_gpd_2022`).
4. **Wikidata Query Service (SPARQL)**: Baseline count of GLAM institutions mapped on the Semantic Web (`glam_count_wikidata`).
5. **Statistics Portugal (INE / Official Census)**: Offline administrative registry used as a validation benchmark to resolve open-data under-sampling (`portugal_glam_census.csv`).


In [1]:
# Install libraries

!pip install SPARQLWrapper rdflib eurostat plotly seaborn

In [2]:
# Import libraries
import requests
import json
import time
import pandas as pd
import eurostat
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from SPARQLWrapper import SPARQLWrapper, JSON

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:
# Mapping countries
countries = {
    'IT': {'name_en': 'Italy','wd_id': 'Q38'},
    'DE': {'name_en': 'Germany', 'wd_id': 'Q183'},
    'NL': {'name_en': 'Netherlands', 'wd_id': 'Q55'},
    'PT': {'name_en': 'Portugal', 'wd_id': 'Q45'},
    'ES': {'name_en': 'Spain', 'wd_id': 'Q29'},
    'FR': {'name_en': 'France','wd_id': 'Q142'}
}

df_countries = pd.DataFrame.from_dict(countries, orient='index').reset_index()
df_countries.rename(columns={'index': 'iso_code'}, inplace=True)
df_countries

,iso_code,name_en,wd_id
0,IT,Italy,Q38
1,DE,Germany,Q183
2,NL,Netherlands,Q55
3,PT,Portugal,Q45
4,ES,Spain,Q29
5,FR,France,Q142


In [4]:
# Europeana Data Extraction
API_KEY = "raystmumenyl"

europeana_data = []

for iso, info in countries.items():
    url = "https://api.europeana.eu/record/v2/search.json"
    params = {
        'wskey': API_KEY,
        'query': '*',
        'qf': f'COUNTRY:{info["name_en"].lower()}',
        'rows': 0
    }

    response = requests.get(url, params=params)
    if response.status_code == 200:
        totale = response.json().get('totalResults', 0)
        europeana_data.append({'iso_code': iso, 'europeana_total': totale})
    else:
        print(f"Errore per {info['name_it']}: {response.status_code}")

    time.sleep(0.5)

df_europeana = pd.DataFrame(europeana_data).sort_values(
    by='europeana_total',
    ascending=False,
    ignore_index=True
)
df_europeana

,iso_code,europeana_total
0,NL,9204845
1,DE,8701240
2,ES,6581724
3,FR,4724898
4,IT,1832376
5,PT,139858


This count accounts for Tier 0 entries as well, despite them not being visible on the website.

In [5]:
# Public expenditure for culture (% of GDP)
df_exp = eurostat.get_data_df('gov_10a_exp')

df_exp_filtered = df_exp[
    (df_exp['sector'] == 'S13') & #General government
    (df_exp['unit'] == 'PC_GDP') & #Percentage of GDP
    (df_exp['cofog99'] == 'GF08') & #Culture sector
    (df_exp['na_item'] == 'TE') &
    (df_exp['geo\\TIME_PERIOD'].isin(countries.keys()))
][['geo\\TIME_PERIOD', '2022']].rename(columns={'geo\\TIME_PERIOD': 'iso_code', '2022': 'culture_expenditure_gpd_2022'})
df_exp_filtered

,iso_code,culture_expenditure_gpd_2022
782812,DE,1.0
782819,ES,1.2
782822,FR,1.4
782827,IT,0.9
782832,NL,1.1
782835,PT,0.9


In [6]:
# Wikidata GLAM institutes for countries
sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.agent = "info-viz-student-project/1.0 (mailto:martina.uccheddu@studio.unibo.it)"
sparql.setReturnFormat(JSON)

countries_list = [(iso, info['wd_id'], info['name_en']) for iso, info in countries.items()]

BATCH_SIZE = 3
wikidata_rows = []

def create_batch(list, dimension):
    for i in range(0, len(list), dimension):
        yield list[i:i + dimension]

for index_batch, batch in enumerate(create_batch(countries_list, BATCH_SIZE), start=1):
    nomi_lotto = ", ".join([p[2] for p in batch])

    qid_clean = []
    for p in batch:
        raw_qid = p[1]
        clean_qid = raw_qid.split("/")[-1].replace("wd:", "")
        qid_clean.append(f"wd:{clean_qid}")

    string_values = " ".join(qid_clean)

    # museums, libraries, archives, galleries
    query_batch = f"""
    SELECT ?country (COUNT(DISTINCT ?item) AS ?count) WHERE {{
      VALUES ?country {{ {string_values} }}
      VALUES ?type {{ wd:Q33506 wd:Q7075 wd:Q166118 wd:Q1007870 }}

      ?item wdt:P31 ?type ;
            wdt:P17 ?country .
    }}
    GROUP BY ?country
    """

    sparql.setQuery(query_batch)

    try:
        results = sparql.query().convert()

        count_temp = {}
        for row in results["results"]["bindings"]:
            qid = row["country"]["value"].split("/")[-1]
            count_temp[qid] = int(row["count"]["value"])

        for iso, qid, nome in batch:
            clean_qid = qid.split("/")[-1].replace("wd:", "")
            values = count_temp.get(clean_qid, 0)
            wikidata_rows.append({'iso_code': iso, 'glam_count_wikidata': values})

    except Exception as e:
        print(f"   Batch Error {index_batch}: {e}")
        for iso, qid, nome in batch:
            wikidata_rows.append({'iso_code': iso, 'glam_count_wikidata': 0})

    if index_batch * BATCH_SIZE < len(countries_list):
        time.sleep(1.5)

df_wikidata = pd.DataFrame(wikidata_rows)
df_wikidata


,iso_code,glam_count_wikidata
0,IT,14236
1,DE,9371
2,NL,1539
3,PT,554
4,ES,2813
5,FR,2639


In [7]:
# Europeana providers
API_KEY = "raystmumenyl"

providers_data = []

for iso, info in countries.items():

    url = "https://api.europeana.eu/record/v2/search.json"

    params = {
        'wskey': API_KEY,
        'query': '*',
        'qf': f'COUNTRY:{info["name_en"].lower()}',
        'rows': 0,
        'profile': 'facets',
        'facet': 'DATA_PROVIDER',
        'f.DATA_PROVIDER.facet.limit': 1500
    }

    try:
        response = requests.get(url, params=params, timeout=20)

        if response.status_code == 200:
            data = response.json()
            facets = data.get('facets', [])

            provider_count = 0
            for facet in facets:
                if facet.get('name') == 'DATA_PROVIDER':
                    provider_count = len(facet.get('fields', []))
                    break

            providers_data.append({'iso_code': iso, 'europeana_providers': provider_count})

        else:
            errore_msg = response.json().get('error', response.text)
            print(f"   Error HTTP {response.status_code}: {errore_msg}")
            providers_data.append({'iso_code': iso, 'europeana_providers': 0})

    except Exception as e:
        print(f"   Connection error for {info['name_en']}: {e}")
        providers_data.append({'iso_code': iso, 'europeana_providers': 0})

    time.sleep(0.5)

df_providers = pd.DataFrame(providers_data)
df_providers

,iso_code,europeana_providers
0,IT,158
1,DE,375
2,NL,104
3,PT,40
4,ES,275
5,FR,50


In [8]:
# Data integration
df_final = df_countries.merge(df_europeana, on='iso_code') \
                    .merge(df_exp_filtered, on='iso_code') \
                    .merge(df_wikidata, on='iso_code') \
                    .merge(df_providers, on='iso_code')

df_final['mobilitation_rate_glam'] = (df_final['europeana_providers'] / df_final['glam_count_wikidata']) * 100

formatted_df = df_final.style.format({
    'europeana_total': '{:,.0f}',
    'mobilitation_rate_glam': '{:.2f}%',
    'culture_expenditure_gpd_2022': '{:.2f}%'
})

formatted_df

,iso_code,name_en,wd_id,europeana_total,culture_expenditure_gpd_2022,glam_count_wikidata,europeana_providers,mobilitation_rate_glam
0,IT,Italy,Q38,"1,832,376",0.90%,14236,158,1.11%
1,DE,Germany,Q183,"8,701,240",1.00%,9371,375,4.00%
2,NL,Netherlands,Q55,"9,204,845",1.10%,1539,104,6.76%
3,PT,Portugal,Q45,"139,858",0.90%,554,40,7.22%
4,ES,Spain,Q29,"6,581,724",1.20%,2813,275,9.78%
5,FR,France,Q142,"4,724,898",1.40%,2639,50,1.89%


In [9]:
# Visualization Stage 1 - Mobilitation Rate Glam
df_support = df_final[['iso_code', 'mobilitation_rate_glam']].copy()
df_support['Active on Europeana (%)'] = df_support['mobilitation_rate_glam']
df_support['Offline GLAMs (%)'] = 100 - df_support['mobilitation_rate_glam']

df_support = df_support.sort_values(by='mobilitation_rate_glam', ascending=False)

df_tidy = df_support.melt(
    id_vars=['iso_code', 'mobilitation_rate_glam'],
    value_vars=['Active on Europeana (%)', 'Offline GLAMs (%)'],
    var_name='Status',
    value_name='Percentage'
)

fig1 = px.bar(
    df_tidy,
    x='iso_code',
    y='Percentage',
    color='Status',
    title="Stage 1: The Mobilization Gap: distribution of active europeana providers",
    labels={
        'iso_code': 'Country',
        'Percentage': 'Share of Physical GLAMs (%)',
        'Status': 'Institutional Status'
    },
    color_discrete_map={
        'Active on Europeana (%)': '#5A5A5A',
        'Offline GLAMs (%)': '#AACAE0'
    }
)

fig1.update_traces(
    texttemplate='',
    hoverinfo='none'
)

df_active = df_tidy[df_tidy['Status'] == 'Active on Europeana (%)']

fig1.add_trace(
    go.Scatter(
        x=df_active['iso_code'],
        y=df_active['Percentage'],
        text=df_active['Percentage'].map('{:.1f}%'.format),
        mode='text',
        textposition='top center',
        textfont=dict(size=12, color='black'),
        showlegend=False,
        hoverinfo='none'
    )
)

fig1.update_layout(
    template="plotly_white",
    height=600,
    width=800,
    barmode='stack',
    hovermode=False,
    xaxis=dict(showgrid=False),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%",
        range=[0, 108]
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig1.show()

This normalized 100% stacked bar chart explores the proportion of active Europeana data providers relative to the broader baseline of physical GLAM institutions mapped via Wikidata across selected countries. By removing absolute volume differences, the visual distribution highlights a notable disparity: across all observed territories, over 89% of mapped institutions do not currently appear as active contributors to Europeana's Linked Data infrastructure, gettin lost in the sea. This observational pattern suggests that a country's institutional density does not necessarily reflect its active online participation rate.

In [10]:
# Visualization Stage 2 - Culture expenditure
x_vals = df_final['culture_expenditure_gpd_2022'].values
y_vals = df_final['europeana_total'].values

m = np.sum(x_vals * y_vals) / np.sum(x_vals**2)
q = 0

x_line = np.linspace(0, x_vals.max() + 0.05, 100)
y_line = m * x_line

fig2 = px.scatter(
    df_final,
    x="culture_expenditure_gpd_2022",
    y="europeana_total",
    text="iso_code",
    color="iso_code",
    title="Stage 2: Relationship between public funding and total shared items in Europeana",
    labels={
        "culture_expenditure_gpd_2022": "Public Funding in Culture (% of GDP)",
        "europeana_total": "Total Objects Shared on Europeana"
    },
    hover_data={
        "europeana_total": ":,.0f",
        "culture_expenditure_gpd_2022": False,
        "iso_code": False
    }
)

fig2.add_trace(
    go.Scatter(
        x=x_line,
        y=y_line,
        mode="lines",
        name="Global Trendline (OLS)",
        line=dict(dash="dash", color="#FF4B4B", width=2.5),
        showlegend=False,
        hoverinfo="skip"
    )
)

fig2.update_layout(
    template="plotly_white",
    height=550,
    width=750,
    showlegend=False,
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%",
        range=[x_vals.min() - 0.08, x_vals.max() + 0.08]
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        tickformat=","
    )
)

fig2.update_traces(
    selector=dict(mode="markers+text"),
    textposition="top center",
    textfont=dict(size=12, family="sans-serif", color="black")
)


fig2.show()

To explore whether higher public funding in culture (% of GDP) aligns with higher volumes of shared digital objects, we map the two variables using a scatter plot. The trendline serves to understand the missing positive correlation we expected. We observe a wide dispersion of data points, suggesting a null correlation. For instance, France (1.4% GDP) and the Netherlands (1.1% GDP) display noticeably different digital object volumes (4.7M vs. 9.2M objects) despite relatively high spending levels. This distribution indicates that macroeconomic funding percentages alone do not account for the observed variance in online artifact accumulation, suggesting that the funding are still not oriented towards european level sharing of digitalized data.

In [11]:
# Visualization stage 3a- Relationship between investment, glam mobilitation rate, europeana total item for each selected countries
fig3 = px.scatter(
    df_final,
    x="mobilitation_rate_glam",
    y="culture_expenditure_gpd_2022",
    size="europeana_total",
    color="iso_code",
    text="iso_code",
    hover_name="name_en",

    hover_data={
        "europeana_total": ":,",
        "mobilitation_rate_glam": ":.2f",
        "culture_expenditure_gpd_2022": ":.2f%",
        "iso_code": False
    },

    title="Stage 3a: European(a) Digital Heritage: relationship between investment,<br> glam mobilitation rate and total shared items</br>",
    labels={
        "mobilitation_rate_glam": "GLAM Mobilitation Rate (% of physical institutions active online)",
        "culture_expenditure_gpd_2022": "Public Funding in Culture (% of GDP)",
        "europeana_total": "Total Objects in Europeana",
    },
    size_max=65
)

fig3.update_layout(
    template="plotly_white",
    height=800,
    width=750,
    showlegend=False,
    margin=dict(t=80),
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    )
)

fig3.update_traces(
    textposition="top center",
    textfont=dict(size=12, family="sans-serif", color="black")
)

fig3.show()



This multivariate bubble chart simultaneously explores public funding (Y-axis), observed mobilization rate (X-axis), and total digital volume (bubble size). In this initial distribution, calculated using crowdsourced Wikidata counts as the baseline denominator, Portugal (PT) appears positioned at a relatively high mobilization rate (8.70%). We note that the baseline Wikidata count for Portugal is noticeably smaller (460 institutions) than that of other nations, prompting an exploratory sensitivity check on how denominator definitions influence comparative metrics.

In [12]:
# Stage 3b: Update with data from portugal GLAM census
df_ine = pd.read_csv('data/portugal_glam_census.csv')
pt_real_glam_total = df_ine['Value'].sum()
df_final.loc[df_final['iso_code'] == 'PT', 'glam_count_wikidata'] = pt_real_glam_total

df_final['mobilitation_rate_glam'] = (df_final['europeana_providers'] / df_final['glam_count_wikidata']) * 100

fig3 = px.scatter(
    df_final,
    x="mobilitation_rate_glam",
    y="culture_expenditure_gpd_2022",
    size="europeana_total",
    color="iso_code",
    text="iso_code",
    hover_name="name_en",
    hover_data={
        "europeana_total": ":,",
        "mobilitation_rate_glam": ":.2f",
        "culture_expenditure_gpd_2022": ":.2f%",
        "iso_code": False
    },
    title="Stage 3b: European(a) Digital Heritage: relationship between investment, <br> glam mobilitation rate (updated) and total shared items </br>",
    labels={
        "mobilitation_rate_glam": "GLAM Mobilitation Rate (% of physical institutions active online)",
        "culture_expenditure_gpd_2022": "Public Funding in Culture (% of GDP)",
        "europeana_total": "Total Objects in Europeana",
    },
    size_max=65
)

fig3.update_layout(
    template="plotly_white",
    height=800,
    width=750,
    showlegend=False,
    margin=dict(t=80),
    xaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor="lightgray",
        ticksuffix="%"
    )
)

fig3.update_traces(
    textposition="top center",
    textfont=dict(size=12, family="sans-serif", color="black")
)

fig3.show()



To better observe the correlation, we replace Portugal's Wikidata baseline (460 institutions) with administrative census data from Statistics Portugal (INE: 3,395 institutions). Maintaining an identical spatial grid (range=[-0.5, 12.5]), we observe a noticeable horizontal shift: Portugal's observed mobilization rate adjusts from 8.70% to 1.18%. This visual variance help us better investigate the correlation: a positive correlation in term of mobilitation rate of GLAM institution on Europeana is shown. Spain, unexpected negative, has a smaller number of total europeana item, showing the investment is not directly correlated to the % of investments. France stand outside this correlation, the country invest more then all the other observed countries but its mobilitation rate shows that a small only of GLAM institution shares data on Europeana. Italy proves to be an expected negative: small investment, small distrubution of providers over total GLAM institutions and, consequentially, small number of total items on Europeana.